In [1]:
import pandas as pd
import numpy as np


## Data raw

In [2]:
data_raw = pd.read_csv("../data/globalterrorismdb_0718dist.csv", encoding="latin1", low_memory=False)

In [3]:
print(f'Rows: {data_raw.shape[0]}, columns: {data_raw.shape[1]}')

Rows: 181691, columns: 135


In [4]:
# список колонок, с указанием количества NaN

for i, col in enumerate(data_raw.columns, 1):
    n_nan = data_raw[col].isna().sum()
    n_zero = (data_raw[col] == 0).sum()
    
    print(f"{i:3} {col:25}  NaN: {n_nan:6}  Zeros: {n_zero:6}")

  1 eventid                    NaN:      0  Zeros:      0
  2 iyear                      NaN:      0  Zeros:      0
  3 imonth                     NaN:      0  Zeros:     20
  4 iday                       NaN:      0  Zeros:    891
  5 approxdate                 NaN: 172452  Zeros:      0
  6 extended                   NaN:      0  Zeros: 173452
  7 resolution                 NaN: 179471  Zeros:      0
  8 country                    NaN:      0  Zeros:      0
  9 country_txt                NaN:      0  Zeros:      0
 10 region                     NaN:      0  Zeros:      0
 11 region_txt                 NaN:      0  Zeros:      0
 12 provstate                  NaN:    421  Zeros:      0
 13 city                       NaN:    435  Zeros:      0
 14 latitude                   NaN:   4556  Zeros:      0
 15 longitude                  NaN:   4557  Zeros:      0
 16 specificity                NaN:      6  Zeros:      0
 17 vicinity                   NaN:      0  Zeros: 168932
 18 location  

In [5]:
# колонки с >10% NaN

cols_nan_10 = data_raw.columns[data_raw.isna().mean() > 0.10]
print(cols_nan_10.tolist())

['approxdate', 'resolution', 'location', 'summary', 'alternative', 'alternative_txt', 'attacktype2', 'attacktype2_txt', 'attacktype3', 'attacktype3_txt', 'corp1', 'targtype2', 'targtype2_txt', 'targsubtype2', 'targsubtype2_txt', 'corp2', 'target2', 'natlty2', 'natlty2_txt', 'targtype3', 'targtype3_txt', 'targsubtype3', 'targsubtype3_txt', 'corp3', 'target3', 'natlty3', 'natlty3_txt', 'gsubname', 'gname2', 'gsubname2', 'gname3', 'gsubname3', 'motive', 'guncertain2', 'guncertain3', 'nperps', 'nperpcap', 'claimed', 'claimmode', 'claimmode_txt', 'claim2', 'claimmode2', 'claimmode2_txt', 'claim3', 'claimmode3', 'claimmode3_txt', 'compclaim', 'weapsubtype1', 'weapsubtype1_txt', 'weaptype2', 'weaptype2_txt', 'weapsubtype2', 'weapsubtype2_txt', 'weaptype3', 'weaptype3_txt', 'weapsubtype3', 'weapsubtype3_txt', 'weaptype4', 'weaptype4_txt', 'weapsubtype4', 'weapsubtype4_txt', 'weapdetail', 'nkillus', 'nkillter', 'nwoundus', 'nwoundte', 'propextent', 'propextent_txt', 'propvalue', 'propcomment', 

In [6]:
# оставляем только колонки, где меньше 10% пропущенных значений

cols_valid = [col for col in data_raw.columns if col not in cols_nan_10]
print(cols_valid)

['eventid', 'iyear', 'imonth', 'iday', 'extended', 'country', 'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude', 'longitude', 'specificity', 'vicinity', 'crit1', 'crit2', 'crit3', 'doubtterr', 'multiple', 'success', 'suicide', 'attacktype1', 'attacktype1_txt', 'targtype1', 'targtype1_txt', 'targsubtype1', 'targsubtype1_txt', 'target1', 'natlty1', 'natlty1_txt', 'gname', 'guncertain1', 'individual', 'weaptype1', 'weaptype1_txt', 'nkill', 'nwound', 'property', 'ishostkid', 'dbsource', 'INT_LOG', 'INT_IDEO', 'INT_MISC', 'INT_ANY']


In [7]:
# убираем колонки с кодовыми значениями, если есть соответствующие текстовые

cols_to_drop = [col for col in cols_valid if col + "_txt" in cols_valid]

## Data filtered

In [8]:
data_filtered = data_raw.drop(columns=cols_to_drop)

In [9]:
# информация по пригодным для EDA колонкам

# column groups
num_cols = data_filtered.select_dtypes(include='number').columns
cat_cols = data_filtered.select_dtypes(include='object').columns
bool_cols = data_filtered.select_dtypes(include='bool').columns
datetime_cols = data_filtered.select_dtypes(include=['datetime', 'datetimetz']).columns

# binary (0/1) columns that are not true boolean
binary_cols = [
    col for col in data_filtered.columns
    if set(data_filtered[col].dropna().unique()) <= {0,1} and col not in bool_cols]

print(f"Numeric columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"True boolean columns: {len(bool_cols)}")
print(f"Binary (0/1) columns: {len(binary_cols)}")
print(f"Datetime columns: {len(datetime_cols)}")

if len(binary_cols) > 0:
    print("\nBinary columns:", binary_cols)

if len(num_cols) > 0:
    print("\nNumeric columns:", num_cols)

if len(cat_cols) > 0:
    print("\nCategorical columns:", cat_cols)

Numeric columns: 70
Categorical columns: 58
True boolean columns: 0
Binary (0/1) columns: 12
Datetime columns: 0

Binary columns: ['extended', 'crit1', 'crit2', 'crit3', 'multiple', 'success', 'suicide', 'guncertain1', 'guncertain2', 'guncertain3', 'individual', 'claim3']

Numeric columns: Index(['eventid', 'iyear', 'imonth', 'iday', 'extended', 'latitude',
       'longitude', 'specificity', 'vicinity', 'crit1', 'crit2', 'crit3',
       'doubtterr', 'alternative', 'multiple', 'success', 'suicide',
       'attacktype2', 'attacktype3', 'targtype2', 'targsubtype2', 'natlty2',
       'targtype3', 'targsubtype3', 'natlty3', 'guncertain1', 'guncertain2',
       'guncertain3', 'individual', 'nperps', 'nperpcap', 'claimed',
       'claimmode', 'claim2', 'claimmode2', 'claim3', 'claimmode3',
       'compclaim', 'weapsubtype1', 'weaptype2', 'weapsubtype2', 'weaptype3',
       'weapsubtype3', 'weaptype4', 'weapsubtype4', 'nkill', 'nkillus',
       'nkillter', 'nwound', 'nwoundus', 'nwoundte', 'pr

In [10]:
# суммируем всех пострадавших (убитых и раненых) в общую категорию victims_total

data_filtered["victims_total"] = data_filtered[["nkill","nwound"]].sum(axis=1)

In [11]:
len(data_filtered.query('imonth == 0'))

20

In [12]:
# убираем события до 1980 и где не отмечен месяц события

data_filtered = data_filtered.query("iyear >= 1980 and imonth != 0")

In [13]:
# собираем финальный список колонок для EDA

cols_filtered = [
    'eventid',
    # даты события
    'iyear', 'imonth', 'iday', 'extended',
    # география
    'country_txt', 'region_txt',  'city', 'latitude', 'longitude',
    # цели
    'targtype1_txt', 'natlty1_txt',
    # атакующие; оружие
    'gname', 'attacktype1_txt', 'weaptype1_txt', 'nperps',
    # потери
    'nkill', 'nwound', 'victims_total', 'nkillter' 
]

## Data clean

In [14]:
data_clean = data_filtered[cols_filtered].copy()
data_clean.columns

Index(['eventid', 'iyear', 'imonth', 'iday', 'extended', 'country_txt',
       'region_txt', 'city', 'latitude', 'longitude', 'targtype1_txt',
       'natlty1_txt', 'gname', 'attacktype1_txt', 'weaptype1_txt', 'nperps',
       'nkill', 'nwound', 'victims_total', 'nkillter'],
      dtype='object')

In [15]:
# переименуем некоторые колонки

rename_map = {
    "eventid": "event_id",
    "iyear": "year",
    "imonth": "month",
    "iday": "day",
    "extended": "is_extended",

    "country_txt": "country",
    "region_txt": "region",

    "targtype1_txt": "target_type",
    "natlty1_txt": "target_nationality",

    "gname": "group_name",
    "attacktype1_txt": "attack_type",
    "weaptype1_txt": "weapon_type",

    "nperps": "perpetrators",
    "nkill": "killed",
    "nwound": "wounded",
    "victims_total": "victims",
    "nkillter": "perp_killed"
}

data_clean = data_clean.rename(columns=rename_map)

### Группировка террористических организаций в более крупные образования

In [16]:
# посмотрим топ огранизаций

print(f"Total groups: {data_clean['group_name'].nunique()}\n")

data_clean['group_name'].value_counts().head(25)

Total groups: 3102



group_name
Unknown                                             80020
Taliban                                              7478
Islamic State of Iraq and the Levant (ISIL)          5613
Shining Path (SL)                                    4553
Farabundo Marti National Liberation Front (FMLN)     3350
Al-Shabaab                                           3288
New People's Army (NPA)                              2746
Boko Haram                                           2418
Revolutionary Armed Forces of Colombia (FARC)        2381
Kurdistan Workers' Party (PKK)                       2310
Communist Party of India - Maoist (CPI-Maoist)       1878
Irish Republican Army (IRA)                          1646
Maoists                                              1629
Liberation Tigers of Tamil Eelam (LTTE)              1602
Basque Fatherland and Freedom (ETA)                  1595
National Liberation Army of Colombia (ELN)           1527
Tehrik-i-Taliban Pakistan (TTP)                      1351
Hou

In [17]:
# варианты написаний и мелкие территориальные деления

print(data_clean.loc[
      data_clean['group_name'].str.contains('al-qaida', case=False, na=False),
      'group_name'].unique())

print(data_clean.loc[
      data_clean['group_name'].str.contains('taliban', case=False, na=False),
      'group_name'].unique())


['Al-Qaida' 'Sympathizers of Al-Qaida Organization'
 'Al-Qaida in Saudi Arabia' 'Al-Qaida in Iraq'
 'Al-Qaida in the Arabian Peninsula (AQAP)'
 'Islambouli Brigades of al-Qaida' 'Al-Qaida in Yemen'
 'Al-Qaida Organization for Jihad in Sweden' 'Al-Qaida in Lebanon'
 'Al-Qaida Network for Southwestern Khulna Division'
 'Al-Qaida in the Islamic Maghreb (AQIM)'
 'Jadid Al-Qaida Bangladesh (JAQB)' 'Al-Qaida Kurdish Battalions (AQKB)'
 'Al-Qaida in the Indian Subcontinent']
['Taliban' 'Taliban (Pakistan)' 'Tehrik-i-Taliban Pakistan (TTP)'
 'Tehrik-e-Taliban Islami (TTI)' 'Punjabi Taliban']


In [18]:
# выделим главные организации и объединим их вариации по маске

def group_major_organizations(df, column="group_name"):

    df = df.copy()

    # сохранить оригинальное название
    df["group_details"] = df[column]

    major_groups = {
        "Taliban": [r"Taliban"],
        "Islamic State": [r"Islamic State", r"\bISIS\b", r"\bISIL\b", r"Daesh"],
        "Al-Qaida": [r"Al[- ]Qaida", r"\bAQAP\b", r"\bAQIM\b"],
        "Boko Haram": [r"Boko Haram"],
        "Al-Shabaab": [r"Al[- ]Shabaab"],
        "FARC": [r"FARC"],
        "Shining Path": [r"Shining Path"],
        "PKK": [r"Kurdistan Workers",r"\bPKK\b"],
        "IRA": [r"Irish Republican Army", r"\bIRA\b", r"Irish Republican"],
        "ETA": [r"Basque Fatherland and Freedom", r"\bETA\b"],
        "LTTE": [r"Liberation Tigers of Tamil Eelam", r"\bLTTE\b"],
        "ELN": [r"National Liberation Army of Colombia", r"\bELN\b"],
        "NPA": [r"New People's Army", r"\bNPA\b" ],
        "FMLN": [r"Farabundo Marti National Liberation Front", r"\bFMLN\b"],
        "Communist / Maoist": [r"Maoist", r"Communist"],
        "Palestinian Militants": [r"Palestinian"],
        "Separatists": [r"Separatist"],
        "Anarchist": [r"Anarchist"],
        "Houthis": [r"Houthi", r"Ansar Allah"],
        "Religious Extremists": [r"Muslim extremists", r"Islamic extremists", 
                                 r"Sunni extremists", r"Salafi extremists"]
    }

    # начальная категория
    df[column] = "Other"

    # Unknown отдельно
    unknown_mask = df["group_details"].isin([
        "Unknown", "Activists", "Demonstrators"
    ])

    df.loc[unknown_mask, column] = "Unknown"

    # классификация
    for major, patterns in major_groups.items():

        pattern = "|".join(patterns)

        mask = (
            df["group_details"].str.contains(
                pattern,
                case=False,
                regex=True,
                na=False
            )
            & (df[column] == "Other")
        )

        df.loc[mask, column] = major

    return df

In [19]:
data_clean = group_major_organizations(data_clean)

In [20]:
# топ организаций после группировки

data_clean['group_name'].value_counts().head(20)

group_name
Unknown                  80025
Other                    36803
Taliban                   8873
Islamic State             7279
Shining Path              4553
Communist / Maoist        4049
FMLN                      3350
Al-Shabaab                3292
NPA                       2746
FARC                      2463
Boko Haram                2418
PKK                       2310
Al-Qaida                  2049
IRA                       1796
Palestinian Militants     1642
LTTE                      1602
ETA                       1595
ELN                       1527
Religious Extremists      1165
Houthis                   1064
Name: count, dtype: int64

In [21]:
# кто вошел в others

data_clean.query("group_name == 'Other'")["group_details"].value_counts().head(20)

group_details
Nicaraguan Democratic Force (FDN)                              895
Manuel Rodriguez Patriotic Front (FPMR)                        830
Sikh Extremists                                                716
Donetsk People's Republic                                      624
African National Congress (South Africa)                       580
Tupac Amaru Revolutionary Movement (MRTA)                      557
Abu Sayyaf Group (ASG)                                         527
Fulani extremists                                              511
Corsican National Liberation Front (FLNC)                      490
M-19 (Movement of April 19)                                    489
People's Liberation Front (JVP)                                433
National Union for the Total Independence of Angola (UNITA)    430
Hamas (Islamic Resistance Movement)                            408
Hezbollah                                                      406
Bangsamoro Islamic Freedom Movement (BIFM)      

In [22]:
# всего мелких огранизаций в others

len(data_clean.loc[data_clean['group_name'] == 'Other', 'group_details'].unique())

2884

## Summary

In [23]:
# codebook с описанием колонок
codebook = pd.read_csv("../data/codebook_columns.csv")

# оставим только колонки, которые вошли в список отфильтрованных
codebook = codebook[codebook["column"].isin(cols_filtered)]

# переименовать
codebook["column"] = codebook["column"].replace(rename_map)

# новые созданные колонки
custom_descriptions = {
    "victims": "Total Victims (killed + wounded)",
    "group_details": "Original group name"}

codebook = pd.concat([
    codebook,
    pd.DataFrame({
        "column": custom_descriptions.keys(),
        "description": custom_descriptions.values()
    })], ignore_index=True)

# информация о датасете
summary = pd.DataFrame({
    "column": data_clean.columns,
    "dtype": data_clean.dtypes.values,
    "NaN": data_clean.isna().sum().values
})

# описание из codebook
summary = summary.merge(codebook, on="column", how="left")

# порядок колонок
summary = summary[["column", "description", "dtype", "NaN"]]

summary

,column,description,dtype,NaN
0,event_id,Unique Event ID,int64,0
1,year,Year of Incident,int64,0
2,month,Month of Incident,int64,0
3,day,Day of Incident,int64,0
4,is_extended,Extended Incident,int64,0
5,country,Country Name,object,0
6,region,Region Name,object,0
7,city,City,object,435
8,latitude,Latitude,float64,4247
9,longitude,Longitude,float64,4248


In [24]:
# структура регионов по странам

for region, countries in data_clean.groupby("region")["country"].unique().items():
    print(f"\n{region}:")
    print(", ".join(sorted(countries)))


Australasia & Oceania:
Australia, Fiji, French Polynesia, New Caledonia, New Hebrides, New Zealand, Papua New Guinea, Solomon Islands, Vanuatu, Wallis and Futuna

Central America & Caribbean:
Antigua and Barbuda, Bahamas, Barbados, Belize, Costa Rica, Cuba, Dominica, Dominican Republic, El Salvador, Grenada, Guadeloupe, Guatemala, Haiti, Honduras, Jamaica, Martinique, Nicaragua, Panama, St. Kitts and Nevis, St. Lucia, Trinidad and Tobago

Central Asia:
Armenia, Azerbaijan, Georgia, Kazakhstan, Kyrgyzstan, Tajikistan, Turkmenistan, Uzbekistan

East Asia:
China, Hong Kong, Japan, Macau, North Korea, South Korea, Taiwan

Eastern Europe:
Albania, Belarus, Bosnia-Herzegovina, Bulgaria, Croatia, Czech Republic, Czechoslovakia, East Germany (GDR), Estonia, Hungary, Kosovo, Latvia, Lithuania, Macedonia, Moldova, Montenegro, Poland, Romania, Russia, Serbia, Serbia-Montenegro, Slovak Republic, Slovenia, Soviet Union, Ukraine, Yugoslavia

Middle East & North Africa:
Algeria, Bahrain, Egypt, Inte

In [25]:
# временные рамка датасета

data_clean['year'].min(), data_clean['year'].max()

(1980, 2017)

In [26]:
data_clean.to_csv("../data/terrorism_clean.csv", index=False)